<a href="https://colab.research.google.com/github/ali-liyakat/PyTorch_for_Deep_Learning/blob/main/03_training_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PyTorch Training Pipeline

In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

### Breast Cancer Dataset

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.shape

(569, 33)

In [4]:
df.drop(columns=['id', 'Unnamed: 32'], inplace= True)

### Train Test Split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2, random_state=42)

### Scaling

In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Label Encoding

In [7]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

### Numpy Arrays to PyTorch Tensors

In [8]:
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [9]:
X_train_tensor

tensor([[-1.4408, -0.4353, -1.3621,  ...,  0.9320,  2.0972,  1.8865],
        [ 1.9741,  1.7330,  2.0917,  ...,  2.6989,  1.8912,  2.4978],
        [-1.4000, -1.2496, -1.3452,  ..., -0.9702,  0.5976,  0.0579],
        ...,
        [ 0.0488, -0.5550, -0.0651,  ..., -1.2390, -0.7086, -1.2715],
        [-0.0390,  0.1021, -0.0314,  ...,  1.0500,  0.4343,  1.2134],
        [-0.5486,  0.3133, -0.6035,  ..., -0.6110, -0.3345, -0.8463]],
       dtype=torch.float64)

### Model

In [10]:
class BreastCancerModel():
    def __init__(self, X):
        self.weights = torch.rand(X.shape[1], 1, dtype=torch.float64, requires_grad=True)
        self.bias = torch.zeros(1, dtype=torch.float64, requires_grad=True)

    def forward(self, X):
        z = torch.matmul(X, self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred

    def loss_function(self, y_pred, y):
      # Clamp predictions to avoid log(0)
      epsilon = 1e-7
      y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

      # Calculate loss
      loss = -(y_train_tensor * torch.log(y_pred) + (1 - y_train_tensor) * torch.log(1 - y_pred)).mean()
      return loss

### Parameters

In [11]:
learning_rate = 0.01
epochs = 25

### Training Pipeline

In [12]:
# create model
model = BreastCancerModel(X_train_tensor)

# define loop
for epoch in range(epochs):

    # forward Pass
    y_pred = model.forward(X_train_tensor)

    # loss calculate
    loss = model.loss_function(y_pred, y_train_tensor)

    # backward Pass
    loss.backward()

    # parameter update
    with torch.no_grad():
      model.weights -= learning_rate * model.weights.grad
      model.bias -= learning_rate * model.bias.grad

    # zero gradients
    model.weights.grad.zero_()
    model.bias.grad.zero_()

    print(f'Epoch: {epoch+1}, Loss: {loss.item()}')

Epoch: 1, Loss: 3.5388065428977273
Epoch: 2, Loss: 3.5258154126599597
Epoch: 3, Loss: 3.5127865129574225
Epoch: 4, Loss: 3.4993748787068086
Epoch: 5, Loss: 3.4859743627796433
Epoch: 6, Loss: 3.4725850960294604
Epoch: 7, Loss: 3.4592072099629085
Epoch: 8, Loss: 3.4458408366899795
Epoch: 9, Loss: 3.4324861089479293
Epoch: 10, Loss: 3.419143160074279
Epoch: 11, Loss: 3.405801741588008
Epoch: 12, Loss: 3.392256148330609
Epoch: 13, Loss: 3.3787229139610813
Epoch: 14, Loss: 3.365139999633758
Epoch: 15, Loss: 3.351414056405832
Epoch: 16, Loss: 3.337701064531956
Epoch: 17, Loss: 3.324001168527016
Epoch: 18, Loss: 3.310314513643038
Epoch: 19, Loss: 3.2965532017125847
Epoch: 20, Loss: 3.2826367406894468
Epoch: 21, Loss: 3.2686979841257218
Epoch: 22, Loss: 3.2544974350169054
Epoch: 23, Loss: 3.2401437213132387
Epoch: 24, Loss: 3.225805226131047
Epoch: 25, Loss: 3.211482117211259


In [13]:
model.weights

tensor([[0.1272],
        [0.3647],
        [0.6150],
        [0.5204],
        [0.9172],
        [0.3563],
        [0.4823],
        [0.3411],
        [0.1647],
        [0.7322],
        [0.2476],
        [0.2193],
        [0.3957],
        [0.2530],
        [0.5819],
        [0.5316],
        [0.1244],
        [0.5317],
        [0.5470],
        [0.3884],
        [0.5050],
        [0.6980],
        [0.2322],
        [0.4281],
        [0.5207],
        [0.2463],
        [0.8839],
        [0.3657],
        [0.6743],
        [0.0010]], dtype=torch.float64, requires_grad=True)

In [14]:
model.bias

tensor([-0.0036], dtype=torch.float64, requires_grad=True)

### Evaluation

In [15]:
# model evaluation
with torch.no_grad():
    y_pred = model.forward(X_test_tensor)
    y_pred = (y_pred > 0.9).float()

    accuaracy = (y_pred == y_test_tensor).float().mean()
    print(f'Accuracy: {accuaracy.item()}')

Accuracy: 0.5452446937561035
